In [ ]:
import numpy as np
import os
import sys

sys.path.append('../../..')
path=os.path.abspath(os.getcwd())

import gc
import jax
import jax.numpy as jnp
from jax import vmap, jit
import fire
import torch
import torch.optim.swa_utils as swa_utils
import torchcde
import torchsde
from torch import nn

from torch import profiler
from tqdm import tqdm
from typing import Sequence


from scipy.linalg import cholesky, inv
is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

from core.utils import lqr, Kgain, batchsize_selector, lqr_batch_jax_heun, generate_system_canon, vt
from configs.lqr_cfg import cfg

In [ ]:
# --- Setup and Physical Parameters ---
dt = cfg.total_time / cfg.nsteps

# Pre-generate canonical system and cost matrices
A_sys, B_sys = generate_system_canon(cfg.N)
Qfc = jnp.eye(cfg.N)
Q = jnp.zeros((cfg.N, cfg.N))
R_dt = jnp.array(cfg.weight * np.eye(B_sys.shape[1]) * dt)

In [ ]:
# JIT the existing Riccati solver
batched_heun = jax.jit(lqr_batch_jax_heun, static_argnums=4)

# Welford's Algorithm accumulators
count = 0
mean = 0.0
M2 = 0.0

pbar = tqdm(range(cfg.iter_count), desc="LQR Evaluation")
# --- Main Evaluation Loop ---
for i in pbar:
    # 1. Prepare Batched System Matrices
    # Randomize the last row of A while keeping canonical structure
    amat = np.random.uniform(-0.5, 0.5, (cfg.batch_size, cfg.N))
    A_batch = np.broadcast_to(A_sys, (cfg.batch_size, cfg.N, cfg.N)).copy()
    A_batch[:, -1, :] = amat
    
    B_batch = np.broadcast_to(B_sys, (cfg.batch_size, B_sys.shape[0], B_sys.shape[1]))

    # 2. Solve Finite-Horizon Riccati (returns optimal gains K)
    # Keeping your original inputs: Q, R/dt, A, B, nsteps, Qfc
    optimal_control_input = batched_heun(Q, R_dt/dt, jnp.array(A_batch), jnp.array(B_batch), int(cfg.nsteps), Qfc)

    # 3. Initial Condition Sampling (Normalized to unit sphere)
    x0_raw = np.random.randn(cfg.batch_size, cfg.N)
    actual_state_x = x0_raw / np.linalg.norm(x0_raw, axis=1, keepdims=True)
    desired_state_xf = np.zeros_like(actual_state_x)
    
    energy = np.zeros(cfg.batch_size)

    # 4. Simulation Loop (Heun's Integration)
    for l in range(cfg.nsteps):
        state_error = actual_state_x - desired_state_xf
        
        # Current Control Action
        action = np.einsum("bij, bj -> bi", optimal_control_input[:, l, :, :], state_error)
        
        # Predictor Step (Euler)
        f1 = np.einsum("bij, bj -> bi", A_batch, actual_state_x) + np.einsum("bij, bj -> bi", B_batch, action)
        pred_state_x = actual_state_x + dt * f1
        
        # Predictor Error and Action
        pred_error = pred_state_x - desired_state_xf
        pred_action = np.einsum("bij, bj -> bi", optimal_control_input[:, l, :, :], pred_error)

        # Corrector Step (Heun)
        f2 = np.einsum("bij, bj -> bi", A_batch, pred_state_x) + np.einsum("bij, bj -> bi", B_batch, pred_action)
        actual_state_x = actual_state_x + (dt / 2.0) * (f1 + f2)

        # Energy Accumulation: u^T * R * u_mid
        u_mid = (action + pred_action) / 2.0
        # R_dt already incorporates weight and dt
        energy += np.einsum("bi, bi -> b", action, np.einsum("ij, bj -> bi", R_dt, u_mid))

    # 5. Terminal Loss (Final state error norm squared)
    terminal_error = np.sum((actual_state_x - desired_state_xf)**2, axis=1)
    
    # 6. Statistical Update (Batch Welford)
    batch_data = energy + terminal_error
    b_count = batch_data.size
    b_mean = batch_data.mean()
    b_M2 = ((batch_data - b_mean) ** 2).sum()

    delta = b_mean - mean
    new_total = count + b_count
    mean += delta * b_count / new_total
    M2 += b_M2 + (delta ** 2) * count * b_count / new_total
    count = new_total
    
    
    current_ste = np.sqrt(M2 / count) / np.sqrt(count) if count > 0 else 0
    pbar.set_postfix({
        "Mean": f"{mean:.5f}",
        "STE": f"{current_ste:.2e}"
    })
    # Cleanup memory
    if i % 100 == 0:
        gc.collect()

# --- Final Output ---
final_ste = np.sqrt(M2 / count) / np.sqrt(count)
print(f"\nFinal Mean Loss: {mean:.6f}")
print(f"Standard Error:  {final_ste:.2e}")